In [1]:
import aiohttp
import asyncio
import nest_asyncio
import os
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple, Union
import json
from github import Github, Auth
from gidgethub.aiohttp import GitHubAPI
from gidgethub import GitHubException
import sys
import base64
from stamina import retry, retry_context
import re
from functools import partial
from textacy import preprocessing
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from qdrant_client.http.models import Distance, VectorParams
import uuid
from searcharray import SearchArray
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import START, StateGraph
from langchain_core.documents import Document
from typing_extensions import List, TypedDict
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# allow for multiple event loops within Jupyter
nest_asyncio.apply()

# load environment variables
load_dotenv()

import pickle

In [2]:
MAX_CHARACTERS = 30_000
NAMESPACE = uuid.NAMESPACE_URL

def strip_markdown(text: str) -> str:
    """Remove common Markdown syntax; keep inner text and emojis."""
    # Headings: ## Title -> Title
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    # Inline code: `code` -> code
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # Bold/italic: **bold** or *italic* -> bold / italic
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", text)
    text = re.sub(r"_{1,2}([^_]+)_{1,2}", r"\1", text)
    # Link markup: [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # Collapse many newlines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def make_normalize_text_pipeline(
    *,
    url_repl: str = "",
    unicode_form: str = "NFC",
):
    """
    Build a textacy pipeline: Markdown/HTML → plain text (emojis kept).

    Args:
        url_repl: Replacement for URLs (default ""). Use "_URL_" to keep a placeholder.
        unicode_form: Unicode normalization form ("NFC", "NFD", "NFKC", "NFKD"). Default "NFC".

    Returns:
        A callable that takes a string and returns preprocessed plain text.
    """
    return preprocessing.make_pipeline(
        strip_markdown,
        preprocessing.remove.html_tags,
        preprocessing.normalize.bullet_points,
        preprocessing.normalize.quotation_marks,
        partial(preprocessing.normalize.unicode, form=unicode_form),
        preprocessing.normalize.whitespace,
    )


# Default pipeline instance
normalize_text_pipeline = make_normalize_text_pipeline()

def repo_to_uuid(repo_name: str) -> str:
    return str(uuid.uuid5(NAMESPACE, repo_name))

def normalize_docs(docs: List[Dict[str, Any]]):
    content_str = ''
    for doc in docs:
        content_str += (doc['content'] + '\n\n')
    truncated = content_str[:MAX_CHARACTERS] if len(content_str) > MAX_CHARACTERS else content_str
    normalized_text = normalize_text_pipeline(truncated)
    return normalized_text


def organize_repo_data(repo_data: List[Dict[Any, Any]]) -> Tuple[List[str], List[Dict[str, Union[str, int]]], List[str]]:
    ids = []
    metadata = []
    docs = []
    for repo in repo_data:
        id = repo_to_uuid(repo['repo'])
        description = repo['description']
        topics = repo['topics']
        doc_source = repo['doc_source']
        stars = repo['stars']
        url = repo['url']
        language = repo['language']
        # append to lists for DB consumption
        ids.append(id)
        metadata.append({
            'id': id,
            'repo': repo['repo'],
            'description': description,
            'topics': topics,
            'language': language,
            'doc_source': doc_source,
            'stars': stars,
            'url': url,
        })
        topics_str = f"Topics: {','.join(topics)}\n"
        docs.append(topics_str + normalize_docs(repo['docs']))
    return ids, metadata, docs


def preprocess_text(text: str | None) -> List[str]:
    """borrowed from Ask Brave search for common preprocessing/tokenizer steps before TF-IDF"""
    if text is None:
        return []
    # Lowercase
    text = text.lower()
    # Remove punctuation and digits
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and short words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

In [3]:
# load cached github data
with open("../data/cached/github_data.pkl", "rb") as f:
    starred_repo_data = pickle.load(f)

In [4]:
ids, metadata, docs = organize_repo_data(starred_repo_data)

In [5]:
metadata[0]

{'id': 'a5ef9890-4c52-5a83-8aa1-c1b266a4f6fb',
 'repo': 'pymc-devs/pytensor',
 'description': 'PyTensor allows you to define, optimize, and efficiently evaluate mathematical expressions involving multi-dimensional arrays.',
 'topics': ['ai',
  'bayesian-inference',
  'computational-science',
  'deep-learning',
  'statistics'],
 'language': 'Python',
 'doc_source': 'readme',
 'stars': 594,
 'url': 'https://github.com/pymc-devs/pytensor'}

In [6]:
docs[0]

'Topics: ai,bayesian-inference,computational-science,deep-learning,statistics\n.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensorRGB.svg\n :height: 100px\n :alt: PyTensor logo\n :align: center\n|Tests Status| |Coverage|\n|Project Name| is a Python library that allows one to define, optimize, and\nefficiently evaluate mathematical expressions involving multi-dimensional arrays.\nIt provides the computational backend for PyMC .\nFeatures\n========\n- A hackable, pure-Python codebase\n- Extensible graph framework suitable for rapid development of custom operators and symbolic optimizations\n- Implements an extensible graph transpilation framework that currently provides\n compilation via C, JAX , and Numba \n- Contrary to PyTorch and TensorFlow, PyTensor maintains a static graph which can be modified in-place to\n allow for advanced optimizations\nGetting started\n===============\n.. code-block:: python\n import pytensor\n from pytensor import tensor as pt\n # D

In [7]:
starred_repo_data[0]

{'repo': 'pymc-devs/pytensor',
 'description': 'PyTensor allows you to define, optimize, and efficiently evaluate mathematical expressions involving multi-dimensional arrays.',
 'topics': ['ai',
  'bayesian-inference',
  'computational-science',
  'deep-learning',
  'statistics'],
 'stars': 594,
 'language': 'Python',
 'url': 'https://github.com/pymc-devs/pytensor',
 'doc_source': 'readme',
 'docs': [{'name': 'README.rst',
   'path': 'README.rst',
   'size': 4199,
   'content': '.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensor_RGB.svg\n    :height: 100px\n    :alt: PyTensor logo\n    :align: center\n\n|Tests Status| |Coverage|\n\n|Project Name| is a Python library that allows one to define, optimize, and\nefficiently evaluate mathematical expressions involving multi-dimensional arrays.\nIt provides the computational backend for `PyMC <https://github.com/pymc-devs/pymc>`__.\n\nFeatures\n========\n\n- A hackable, pure-Python codebase\n- Extensible graph frame

In [8]:
df = pd.DataFrame(starred_repo_data)
df.head()

,repo,description,topics,stars,language,url,doc_source,docs
0,pymc-devs/pytensor,"PyTensor allows you to define, optimize, and e...","[ai, bayesian-inference, computational-science...",594,Python,https://github.com/pymc-devs/pytensor,readme,"[{'name': 'README.rst', 'path': 'README.rst', ..."
1,Pyomo/pyomo-gallery,A collection of Pyomo examples,[],315,Jupyter Notebook,https://github.com/Pyomo/pyomo-gallery,readme,"[{'name': 'README.md', 'path': 'README.md', 's..."
2,Pyomo/pyomo-tutorials,A modern set of Pyomo tutorials,[],41,Python,https://github.com/Pyomo/pyomo-tutorials,readme,"[{'name': 'README.md', 'path': 'README.md', 's..."
3,norhum/reinforcement-learning-from-scratch,RL for total beginners,[],120,Jupyter Notebook,https://github.com/norhum/reinforcement-learni...,readme,"[{'name': 'README.md', 'path': 'README.md', 's..."
4,deanmalmgren/textract,extract text from any document. no muss. no fuss.,"[data-mining, natural-language-processing, pyt...",4464,HTML,https://github.com/deanmalmgren/textract,readme,"[{'name': 'README.rst', 'path': 'README.rst', ..."


In [9]:
df['description_tokenized'] = SearchArray.index(df['description'], tokenizer=preprocess_text)

2026-03-12 03:23:00,384 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-12 03:23:00,385 - searcharray.indexing - INFO - 0 Batch Start tokenization
2026-03-12 03:23:00,386 - searcharray.indexing - INFO - Tokenizing 2049 documents
2026-03-12 03:23:02,006 - searcharray.indexing - INFO - Tokenization -- vstacking
2026-03-12 03:23:02,006 - searcharray.indexing - INFO - Tokenization -- DONE
2026-03-12 03:23:02,007 - searcharray.indexing - INFO - Inverting docs->terms
2026-03-12 03:23:02,008 - searcharray.indexing - INFO - Encoding positions to bit array
2026-03-12 03:23:02,010 - searcharray.indexing - INFO - Batch tokenization complete
2026-03-12 03:23:02,011 - searcharray.indexing - INFO - (main thread) Processing 1 batch results
2026-03-12 03:23:02,012 - searcharray.indexing - INFO - Indexing from tokenization complete


In [10]:
df.head()

,repo,description,topics,stars,language,url,doc_source,docs,description_tokenized
0,pymc-devs/pytensor,"PyTensor allows you to define, optimize, and e...","[ai, bayesian-inference, computational-science...",594,Python,https://github.com/pymc-devs/pytensor,readme,"[{'name': 'README.rst', 'path': 'README.rst', ...","Terms({'evaluate', 'expression', 'pytensor', '..."
1,Pyomo/pyomo-gallery,A collection of Pyomo examples,[],315,Jupyter Notebook,https://github.com/Pyomo/pyomo-gallery,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'example', 'collection', 'pyomo'})"
2,Pyomo/pyomo-tutorials,A modern set of Pyomo tutorials,[],41,Python,https://github.com/Pyomo/pyomo-tutorials,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'modern', 'tutorial', 'set', 'pyomo'})"
3,norhum/reinforcement-learning-from-scratch,RL for total beginners,[],120,Jupyter Notebook,https://github.com/norhum/reinforcement-learni...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'beginner', 'total'})"
4,deanmalmgren/textract,extract text from any document. no muss. no fuss.,"[data-mining, natural-language-processing, pyt...",4464,HTML,https://github.com/deanmalmgren/textract,readme,"[{'name': 'README.rst', 'path': 'README.rst', ...","Terms({'mus', 'fuss', 'document', 'extract', '..."


In [11]:
df['doc_source'].value_counts()

doc_source
readme    2041
Name: count, dtype: int64

In [12]:
df['docs'][0]

[{'name': 'README.rst',
  'path': 'README.rst',
  'size': 4199,
  'content': '.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensor_RGB.svg\n    :height: 100px\n    :alt: PyTensor logo\n    :align: center\n\n|Tests Status| |Coverage|\n\n|Project Name| is a Python library that allows one to define, optimize, and\nefficiently evaluate mathematical expressions involving multi-dimensional arrays.\nIt provides the computational backend for `PyMC <https://github.com/pymc-devs/pymc>`__.\n\nFeatures\n========\n\n- A hackable, pure-Python codebase\n- Extensible graph framework suitable for rapid development of custom operators and symbolic optimizations\n- Implements an extensible graph transpilation framework that currently provides\n  compilation via C, `JAX <https://github.com/google/jax>`__, and `Numba <https://github.com/numba/numba>`__\n- Contrary to PyTorch and TensorFlow, PyTensor maintains a static graph which can be modified in-place to\n  allow for advanced op

In [13]:
df['docs'][0]

[{'name': 'README.rst',
  'path': 'README.rst',
  'size': 4199,
  'content': '.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensor_RGB.svg\n    :height: 100px\n    :alt: PyTensor logo\n    :align: center\n\n|Tests Status| |Coverage|\n\n|Project Name| is a Python library that allows one to define, optimize, and\nefficiently evaluate mathematical expressions involving multi-dimensional arrays.\nIt provides the computational backend for `PyMC <https://github.com/pymc-devs/pymc>`__.\n\nFeatures\n========\n\n- A hackable, pure-Python codebase\n- Extensible graph framework suitable for rapid development of custom operators and symbolic optimizations\n- Implements an extensible graph transpilation framework that currently provides\n  compilation via C, `JAX <https://github.com/google/jax>`__, and `Numba <https://github.com/numba/numba>`__\n- Contrary to PyTorch and TensorFlow, PyTensor maintains a static graph which can be modified in-place to\n  allow for advanced op

In [14]:
# Concatenate content from all docs in the list
df['content'] = df['docs'].apply(lambda x: '\n\n'.join(doc['content'] for doc in x) if x else None)

In [15]:
df[['repo','description','description_tokenized','content']].head()

,repo,description,description_tokenized,content
0,pymc-devs/pytensor,"PyTensor allows you to define, optimize, and e...","Terms({'evaluate', 'expression', 'pytensor', '...",.. image:: https://cdn.rawgit.com/pymc-devs/py...
1,Pyomo/pyomo-gallery,A collection of Pyomo examples,"Terms({'example', 'collection', 'pyomo'})",[![Project Status: Inactive – The project has ...
2,Pyomo/pyomo-tutorials,A modern set of Pyomo tutorials,"Terms({'modern', 'tutorial', 'set', 'pyomo'})",# Pyomo Tutorials\n\nThis repository hosts a m...
3,norhum/reinforcement-learning-from-scratch,RL for total beginners,"Terms({'beginner', 'total'})",# Reinforcement Learning From Scratch\n\nWelco...
4,deanmalmgren/textract,extract text from any document. no muss. no fuss.,"Terms({'mus', 'fuss', 'document', 'extract', '...",.. NOTES FOR CREATING A RELEASE:\n..\n.. * b...


In [16]:
# normalize content
df["content_normalized"] = df["content"].map(
    lambda x: normalize_text_pipeline(x[:MAX_CHARACTERS]) if pd.notna(x) else x
)

In [17]:
df['content_normalized'][10]

"GitButler\n \n \n Git, but better.\n \n GitButler is a modern Git-based version control interface with both a GUI and CLI built from the ground up for AI-powered workflows.\n \n \n Website\n - \n Blog\n - \n Docs\n - \n Downloads\n \n \n \n Our beautiful GUI\n \n Our amazing but CLI\n \n[![TWEET][s1]][l1] [\n![BLUESKY][s8]][l8] [![DISCORD][s2]][l2]\n[![CI][s0]][l0] [![INSTA][s3]][l3] [![YOUTUBE][s5]][l5] [![DEEPWIKI][s7]][l7]\n[s0]: https://github.com/gitbutlerapp/gitbutler/actions/workflows/push.yaml/badge.svg\n[l0]: https://github.com/gitbutlerapp/gitbutler/actions/workflows/push.yaml\n[s1]: https://img.shields.io/badge/Twitter-black?logo=x&logoColor=white\n[l1]: https://twitter.com/intent/follow?screen_name=gitbutler\n[s2]: https://img.shields.io/discord/1060193121130000425?label=Discord&color=5865F2\n[l2]: https://discord.gg/MmFkmaJ42D\n[s3]: https://img.shields.io/badge/Instagram-E4405F?logo=instagram&logoColor=white\n[l3]: https://www.instagram.com/gitbutler/\n[s5]: https://img.

In [18]:
# tokenize content now
df['content_tokenized'] = SearchArray.index(df['content_normalized'], tokenizer=preprocess_text)

2026-03-12 03:23:20,452 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-12 03:23:20,454 - searcharray.indexing - INFO - 0 Batch Start tokenization
2026-03-12 03:23:20,455 - searcharray.indexing - INFO - Tokenizing 2049 documents
2026-03-12 03:23:27,435 - searcharray.indexing - INFO - Tokenization -- vstacking
2026-03-12 03:23:27,438 - searcharray.indexing - INFO - Tokenization -- DONE
2026-03-12 03:23:27,443 - searcharray.indexing - INFO - Inverting docs->terms
2026-03-12 03:23:27,591 - searcharray.indexing - INFO - Encoding positions to bit array
2026-03-12 03:23:27,635 - searcharray.indexing - INFO - Batch tokenization complete
2026-03-12 03:23:27,636 - searcharray.indexing - INFO - (main thread) Processing 1 batch results
2026-03-12 03:23:27,655 - searcharray.indexing - INFO - Indexing from tokenization complete


In [23]:
df['score'] = df['content_tokenized'].array.score("awesome")

In [24]:
df.sort_values('score', ascending=False).head(10)

,repo,description,topics,stars,language,url,doc_source,docs,description_tokenized,content,content_normalized,content_tokenized,score
1533,MarcSkovMadsen/awesome-streamlit,The purpose of this project is to share knowle...,"[analytics, apps, awesome-list, data, data-sci...",2243,HTML,https://github.com/MarcSkovMadsen/awesome-stre...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'project', 'knowledge', 'purpose', 'str...",# Awesome Streamlit [![Awesome](https://cdn.ra...,Awesome Streamlit ![Awesome](https://github.co...,"Terms({'thing', 'vision', 'identify', 'quickly...",2.460909
1516,ramitsurana/awesome-kubernetes,A curated list for awesome kubernetes sources ...,"[aws, azure, books, cloud-providers, deploy-ku...",15825,Shell,https://github.com/ramitsurana/awesome-kubernetes,readme,"[{'name': 'README.md', 'path': 'docs/README.md...","Terms({'curated', 'source', 'list', 'tada', 'k...",Awesome-Kubernetes\n==========================...,Awesome-Kubernetes\n==========================...,"Terms({'alexmt', 'linux', 'link', 'without', '...",2.429229
169,LincolnBurrows2017/gauss-awesome-recommender-s...,gauss-awesome-recommender-system-engine,[],122,Python,https://github.com/LincolnBurrows2017/gauss-aw...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'system', 'recommender', 'engine', 'gau...",# Gauss: Awesome Recommender System Engine 🚀\n...,Gauss: Awesome Recommender System Engine 🚀\n!G...,"Terms({'powerful', 'supervised', 'start', 'rec...",2.401310
1950,jnv/lists,The definitive list of lists (of lists) curate...,"[awesome, awesome-list, curated-lists, lists, ...",11002,None,https://github.com/jnv/lists,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'elsewhere', 'curated', 'definitive', '...","# Lists\n\nList of useful, silly and [awesome]...","Lists\nList of useful, silly and awesome lists...","Terms({'prepare', 'thing', 'endangered', 'dayw...",2.398078
61,sickn33/antigravity-awesome-skills,The Ultimate Collection of 900+ Agentic Skills...,"[agentic-skills, ai-agents, antigravity, auton...",18096,Python,https://github.com/sickn33/antigravity-awesome...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'code', 'claude', 'tested', 'official',...",# 🌌 Antigravity Awesome Skills: 968+ Agentic S...,🌌 Antigravity Awesome Skills: 968+ Agentic Ski...,"Terms({'ledger', 'react', 'start', 'companion'...",2.363399
78,brianspiering/awesome-deep-rl,A curated list of awesome Deep Reinforcement L...,[],151,None,https://github.com/brianspiering/awesome-deep-rl,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'reinforcement', 'curated', 'list', 'le...",Awesome Deep Reinforcement Learning [![Awesome...,Awesome Deep Reinforcement Learning ![Awesome]...,"Terms({'neighboring', 'book', 'mingo', 'public...",2.351770
471,zudochkin/awesome-newsletters,A list of amazing Newsletters,"[awesome, awesome-list, email, email-newslette...",4322,None,https://github.com/zudochkin/awesome-newsletters,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'list', 'amazing', 'newsletter'})","A curated list of newsletters, awesome newslet...","A curated list of newsletters, awesome newslet...","Terms({'french', 'thing', 'loop', 'categorized...",2.345463
1123,mauhai/awesome-jupyterlab,A curated list of awesome JupyterLab extensio...,"[awesome, collections, jupyterlab, jupyterlab-...",2616,None,https://github.com/mauhai/awesome-jupyterlab,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'curated', 'list', 'jupyterlab', 'resou...",# Awesome JupyterLab[![Awesome](https://cdn.ra...,Awesome JupyterLab![Awesome](https://github.co...,"Terms({'nice', 'quickly', 'may', 'intellij', '...",2.320860
739,EthicalML/awesome-annual-reviews-and-trends,A curated list of awesome year-in-review and a...,[],36,None,https://github.com/EthicalML/awesome-annual-re...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'annual', 'trend', 'year', 'curated', '...

In [25]:
awesome_curated_lists = "awesome curated lists"
tokenized_phrase = preprocess_text(awesome_curated_lists)
tokenized_phrase

['awesome', 'curated', 'list']

In [26]:
df['score'] = df['content_tokenized'].array.score(tokenized_phrase)

In [28]:
df.sort_values('score', ascending=False).head(30)

,repo,description,topics,stars,language,url,doc_source,docs,description_tokenized,content,content_normalized,content_tokenized,score
1123,mauhai/awesome-jupyterlab,A curated list of awesome JupyterLab extensio...,"[awesome, collections, jupyterlab, jupyterlab-...",2616,None,https://github.com/mauhai/awesome-jupyterlab,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'curated', 'list', 'jupyterlab', 'resou...",# Awesome JupyterLab[![Awesome](https://cdn.ra...,Awesome JupyterLab![Awesome](https://github.co...,"Terms({'nice', 'quickly', 'may', 'intellij', '...",3.570782
887,shubhamgrg04/awesome-diagramming,A curated collection of diagramming tools used...,"[awesome-list, diagram, uml]",3225,None,https://github.com/shubhamgrg04/awesome-diagra...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'diagramming', 'curated', 'engineering'...",# Awesome Diagramming [![Awesome](https://cdn....,Awesome Diagramming ![Awesome](https://github....,"Terms({'purpose', 'big', 'essential', 'nomnoml...",3.451823
840,hemansnation/deep-learning-project-ideas,"Curated list of Machine Learning, NLP, Vision,...",[],29,None,https://github.com/hemansnation/deep-learning-...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'project', 'vision', 'curated', 'list',...",<!-- markdownlint-disable MD033 -->\n\n# Aweso...,Awesome Deep Learning Project Ideas\n![Awesome...,"Terms({'endangered', 'physionet', 'vision', 'c...",2.704698
1373,wuchong/awesome-flink,😎 A curated list of amazingly awesome Flink a...,"[awesome, awesome-flink, awesome-list, flink]",781,None,https://github.com/wuchong/awesome-flink,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'amazingly', 'curated', 'list', 'ecosys...","[<img src=""https://raw.githubusercontent.com/w...",Awesome Flink ![Awesome](https://github.com/si...,"Terms({'thing', 'foundational', 'collection', ...",2.554654
1019,NirantK/awesome-project-ideas,"Curated list of Machine Learning, NLP, Vision,...","[awesome, awesome-list, classification, datase...",8960,None,https://github.com/NirantK/awesome-project-ideas,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'project', 'vision', 'curated', 'list',...",<!-- markdownlint-disable MD033 -->\n\n# Aweso...,Awesome Deep Learning Project Ideas\n![Awesome...,"Terms({'endangered', 'physionet', 'vision', 'c...",2.509537
1018,costezki/awesome-nlprojects,List of projects related to Natural Language P...,"[awersome-list, machine-learning, natural-lang...",361,None,https://github.com/costezki/awesome-nlprojects,readme,"[{'name': 'readme.md', 'path': 'readme.md', 's...","Terms({'project', 'language', 'list', 'exist',...",# Awesome NLP Projects\n[![Awesome](https://cd...,Awesome NLP Projects\n![Awesome](https://githu...,"Terms({'epilog', 'acquiring', 'chatbots', 'scr...",2.161598
713,kengz/awesome-deep-rl,A curated list of awesome Deep Reinforcement L...,"[awesome-list, deep-reinforcement-learning, de...",872,None,https://github.com/kengz/awesome-deep-rl,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'reinforcement', 'curated', 'list', 'le...",# Awesome Deep RL [![Awesome](https://awesome....,Awesome Deep RL ![Awesome](https://awesome.re)...,"Terms({'vision', 'rlbench', 'robosumo', 'softl...",2.092480
1372,youngwookim/awesome-hadoop,A curated list of amazingly awesome Hadoop and...,[],1114,None,https://github.com/youngwookim/awesome-hadoop,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'amazingly', 'curated', 'list', 'ecosys...",# Awesome Hadoop [![Awesome](https://cdn.rawgi...,Awesome Hadoop ![Awesome](https://github.com/s...,"Terms({'complexity', 'specification', 'scripti...",2.027645
1137,wsvincent/awesome-django,A curated list of awesome things related to Dj...,"[awesome, awesome-list, django]",11012,Python,https://github.com/wsvincent/awesome-django,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'thing', 'curated', 'list', 'django', '...",#

In [32]:
def multi_match_search(query: str, df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Tokenize the query using each column's own tokenizer
    tokenized_queries = {
        col: df[col].array.tokenizer(query)
        for col in columns
    }

    # Score each column for each query term → shape (num_terms x num_docs) per column
    field_scores = {
        col: np.asarray([df[col].array.score(term) for term in tokenized_queries[col]])
        for col in columns
    }

    # Determine the number of terms from the first column
    num_terms = max(len(scores) for scores in field_scores.values())

    # For each term, take the element-wise max score across all columns (dismax)
    best_term_scores_per_doc = []
    for term_idx in range(num_terms):
        term_scores_across_fields = [
            field_scores[col][term_idx]
            for col in columns
            if term_idx < len(field_scores[col])
        ]
        best_score = np.max(term_scores_across_fields, axis=0)
        best_term_scores_per_doc.append(best_score)

    # Sum the best-field scores across all terms for the final document score
    result = df.copy()
    result['score'] = np.sum(best_term_scores_per_doc, axis=0)
    return result.sort_values('score', ascending=False)

In [33]:
results = multi_match_search(
    query="awesome curated lists",
    df=df,
    columns=["description_tokenized", "content_tokenized"]
)

In [36]:
results.head(30)

,repo,description,topics,stars,language,url,doc_source,docs,description_tokenized,content,content_normalized,content_tokenized,score
471,zudochkin/awesome-newsletters,A list of amazing Newsletters,"[awesome, awesome-list, email, email-newslette...",4322,None,https://github.com/zudochkin/awesome-newsletters,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'list', 'amazing', 'newsletter'})","A curated list of newsletters, awesome newslet...","A curated list of newsletters, awesome newslet...","Terms({'french', 'thing', 'loop', 'categorized...",7.204165
1950,jnv/lists,The definitive list of lists (of lists) curate...,"[awesome, awesome-list, curated-lists, lists, ...",11002,None,https://github.com/jnv/lists,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'elsewhere', 'curated', 'definitive', '...","# Lists\n\nList of useful, silly and [awesome]...","Lists\nList of useful, silly and awesome lists...","Terms({'prepare', 'thing', 'endangered', 'dayw...",6.920668
739,EthicalML/awesome-annual-reviews-and-trends,A curated list of awesome year-in-review and a...,[],36,None,https://github.com/EthicalML/awesome-annual-re...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'annual', 'trend', 'year', 'curated', '...",[![Awesome](https://github.com/EthicalML/aweso...,![Awesome](https://github.com/sindresorhus/awe...,"Terms({'find', 'still', 'whole', 'commit', 'bo...",6.203630
1516,ramitsurana/awesome-kubernetes,A curated list for awesome kubernetes sources ...,"[aws, azure, books, cloud-providers, deploy-ku...",15825,Shell,https://github.com/ramitsurana/awesome-kubernetes,readme,"[{'name': 'README.md', 'path': 'docs/README.md...","Terms({'curated', 'source', 'list', 'tada', 'k...",Awesome-Kubernetes\n==========================...,Awesome-Kubernetes\n==========================...,"Terms({'alexmt', 'linux', 'link', 'without', '...",6.136012
1961,visenger/awesome-mlops,A curated list of references for MLOps,"[ai, data-science, devops, engineering, federa...",13717,None,https://github.com/visenger/awesome-mlops,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'reference', 'list', 'mlops', 'curated'})",# Awesome MLOps [![Awesome](https://cdn.rawgit...,Awesome MLOps ![Awesome](https://github.com/si...,"Terms({'thing', 'loop', 'leanpub', 'specificat...",6.003039
1881,pditommaso/awesome-pipeline,A curated list of awesome pipeline toolkits in...,"[awesome-list, workflow]",6533,None,https://github.com/pditommaso/awesome-pipeline,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'inspired', 'sysadmin', 'curated', 'lis...",Awesome Pipeline\n================\n\nA curate...,Awesome Pipeline\n================\nA curated ...,"Terms({'thing', 'loop', 'scripting', 'react', ...",5.927334
1123,mauhai/awesome-jupyterlab,A curated list of awesome JupyterLab extensio...,"[awesome, collections, jupyterlab, jupyterlab-...",2616,None,https://github.com/mauhai/awesome-jupyterlab,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'curated', 'list', 'jupyterlab', 'resou...",# Awesome JupyterLab[![Awesome](https://cdn.ra...,Awesome JupyterLab![Awesome](https://github.co...,"Terms({'nice', 'quickly', 'may', 'intellij', '...",5.893960
1374,pabloem/awesome-beam,A curated list of awesome resources for Apache...,[],145,None,https://github.com/pabloem/awesome-beam,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'curated', 'list', 'apache', 'beam', 'r...","[<img src=""https://beam.apache.org/images/logo...",Awesome Beam ![Awesome](https://github.com/sin...,"Terms({'thing', 'powerful', 'kirpichov', 'big'...",5.834331
1874,theanalyst/awesome-distributed-systems,A curated list to learn about distributed systems,"[architecture, consensus, distributed-systems,...",11578,None,https://github.com/theanalyst/awesome-distribu...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...","Terms({'distributed', 'curated', 'learn', 'lis...",# awesome-distributed-syste

In [37]:
## TODO: tokenize more fields and search with multi-match-search function